## Open-Sora on Google Colab with TPU, Gradio Interface and Ngrok Tunnel

**Experimental Notebook - Work in Progress**

This notebook attempts to run Open-Sora for video generation from text prompts using Google Colab's TPU resources. It aims to provide a Gradio web interface and use Ngrok for public access.

**Important Notes for TPU Usage:**
1.  **Runtime Type:** Make sure to change your Colab runtime to TPU (`Runtime -> Change runtime type -> Hardware accelerator: TPU`).
2.  **PyTorch/XLA:** This notebook uses PyTorch/XLA for TPU compatibility. Not all PyTorch operations or third-party libraries (especially those optimized for CUDA) are guaranteed to work seamlessly.
3.  **CUDA-specific Libraries:** Libraries like xformers, Apex, and FlashAttention are designed for NVIDIA GPUs and will likely **not** work on TPUs. This notebook will attempt to run Open-Sora without them, which might affect performance or require code modifications in Open-Sora itself if it heavily relies on them.
4.  **Experimental:** This is an experimental setup. Success is not guaranteed, and significant debugging or code changes within Open-Sora's source might be needed.

### 1. Install PyTorch/XLA and Other Dependencies

In [ ]:
# Install PyTorch/XLA (for TPU support)
# This command is specific to Colab environments with TPUs.
!pip install cloud-tpu-client==0.10 torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 torch_xla[tpu]~=2.1.0 --quiet

# Install other base dependencies for Open-Sora 
# Note: CUDA-specific libraries like xformers, apex, flash-attention are excluded as they won't work on TPU.
!pip install ninja colossalai mmengine gradio pyngrok --quiet

# Verification (optional)
print("Verifying PyTorch/XLA installation...")
import torch
import torch_xla.core.xla_model as xm
try:
    if xm.xla_device():
        print(f"PyTorch/XLA successfully connected to TPU: {xm.xla_real_devices([str(xm.xla_device())])}")
    else:
        print("PyTorch/XLA device not found. Make sure runtime is set to TPU.")
except Exception as e:
    print(f"Error during PyTorch/XLA verification: {e}")

### 2. Clone Open-Sora Repository and Install It (Editable Mode)

In [ ]:
%cd /content
!git clone https://github.com/hpcaitech/Open-Sora
%cd /content/Open-Sora
# Attempt to install Open-Sora. We might need to modify its setup.py or requirements 
# if it strictly demands CUDA-specific packages that were not installed.
!pip install -v -e .
print("Open-Sora cloned and installation attempted.")

### 3. Download Pre-trained Models

In [ ]:
!apt -y install -qq aria2

# Download the 16x256x256 model (smaller, faster for testing)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/hpcai-tech/Open-Sora/resolve/main/OpenSora-v1-16x256x256.pth -d /content/Open-Sora/models -o OpenSora-v1-16x256x256.pth

# (Optional) Download a higher quality model (e.g., 16x512x512) - uncomment if you have Colab Pro and more time
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/hpcai-tech/Open-Sora/resolve/main/OpenSora-v1-HQ-16x512x512.pth -d /content/Open-Sora/models -o OpenSora-v1-HQ-16x512x512.pth

# Download T5 text encoder model
!git clone https://huggingface.co/DeepFloyd/t5-v1_1-xxl /content/Open-Sora/pretrained_models/t5_ckpts/t5-v1_1-xxl
print("Pre-trained models downloaded.")

### 4. Adapt Open-Sora's Inference for TPU

This is the most complex part and may require significant trial and error. We will attempt to:
1.  Load the model and necessary components.
2.  Move them to the XLA device.
3.  Modify the inference call to work in a TPU context, excluding CUDA-specific optimizations (like xformers/flash-attention which are not installed).

**Note:** The following code is a **starting point** and will likely need debugging and adjustments based on Open-Sora's internal structure and PyTorch/XLA's behavior.

In [ ]:
import torch
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import os
import sys
import subprocess
import time # Ensure time is imported

os.chdir('/content/Open-Sora') # Ensure we are in the correct directory
sys.path.append('/content/Open-Sora') # Add to path to import its modules

VIDEO_OUTPUT_DIR_TPU = '/content/Open-Sora/samples_tpu'
if not os.path.exists(VIDEO_OUTPUT_DIR_TPU):
    os.makedirs(VIDEO_OUTPUT_DIR_TPU)

# Attempt to import necessary parts from Open-Sora. This might fail if there are hard CUDA dependencies at import time.
from opensora.models.diffusion.latte.modeling_latte import LatteT2V
from opensora.models.text_encoder.t5 import T5Encoder
from opensora.utils.config_utils import Config, load_config
from opensora.utils.misc import to_torch_dtype
from opensora.models.ae import get_ae_model # This might need to be replaced if it's CUDA-specific
from torchvision import transforms
from diffusers.schedulers import (DDIMScheduler, DDPMScheduler, PNDMScheduler, 
                                  EulerDiscreteScheduler, DPMSolverMultistepScheduler, 
                                  HeunDiscreteScheduler, EulerAncestralDiscreteScheduler, 
                                  DEISMultistepScheduler, KDPM2AncestralDiscreteScheduler)
from diffusers.models import AutoencoderKL
import PIL.Image

TPU_DEVICE = xm.xla_device() # Get the XLA device
print(f"Using XLA device: {TPU_DEVICE}")

MODEL_OPTIONS_TPU = {
    "16x256x256 (Fastest - TPU)": {
        "config_path": "configs/opensora/inference/16x256x256.py",
        "ckpt_path": "/content/Open-Sora/models/OpenSora-v1-16x256x256.pth"
    }
    # Add more if other models are downloaded and configs exist
}

def generate_video_tpu(prompt_text, model_choice_tpu):
    if not prompt_text:
        return None, "Error: Prompt cannot be empty."

    selected_model_info = MODEL_OPTIONS_TPU.get(model_choice_tpu)
    if not selected_model_info:
        return None, f"Error: Model {model_choice_tpu} not found."

    config_file_path = selected_model_info["config_path"]
    model_ckpt_path = selected_model_info["ckpt_path"]

    if not os.path.exists(model_ckpt_path):
        return None, f"Error: Model checkpoint {model_ckpt_path} not found."

    try:
        # Load configuration
        args = load_config(config_file_path) 
        print(f"Config loaded: {args}")

        dtype = to_torch_dtype(args.dtype if hasattr(args, 'dtype') else 'fp32') 
        print(f"Using DType: {dtype}")

        print("Loading VAE...")
        vae = AutoencoderKL.from_pretrained(args.vae_path).to(TPU_DEVICE).to(dtype)
        print("VAE loaded and moved to TPU.")

        print("Loading Text Encoder...")
        text_encoder = T5Encoder(args.text_encoder_path, 
                                 model_max_length=args.model_max_length if hasattr(args, 'model_max_length') else 200, 
                                 text_embed_dim=args.text_embed_dim if hasattr(args, 'text_embed_dim') else args.text_encoder_output_dim
                                ).to(TPU_DEVICE).to(dtype) 
        print("Text Encoder loaded and moved to TPU.")

        print("Loading Diffusion Model (LatteT2V)...")
        model_args = {
            'num_frames': args.num_frames,
            'image_size': args.image_size,
            'num_classes': args.num_classes if hasattr(args, 'num_classes') else 0,
            'learn_sigma': args.learn_sigma if hasattr(args, 'learn_sigma') else True,
            'in_channels': args.in_channels if hasattr(args, 'in_channels') else vae.config.latent_channels,
            'text_embed_dim': args.text_embed_dim if hasattr(args, 'text_embed_dim') else args.text_encoder_output_dim,
            'patch_size': args.patch_size if hasattr(args, 'patch_size') else 1,
            'depth': args.depth if hasattr(args, 'depth') else 28,
            'hidden_size': args.hidden_size if hasattr(args, 'hidden_size') else 1152, 
            'num_heads': args.num_heads if hasattr(args, 'num_heads') else 16,
            'mlp_ratio': args.mlp_ratio if hasattr(args, 'mlp_ratio') else 4.0,
            'use_flash_attn': False, 
            'use_sdpa': False, 
            'force_images': args.force_images if hasattr(args, 'force_images') else False,
            'text_dropout_prob': args.text_dropout_prob if hasattr(args, 'text_dropout_prob') else 0.1
        }
        if hasattr(args, 'qk_norm'): model_args['qk_norm'] = args.qk_norm

        latte_model = LatteT2V(**model_args).to(TPU_DEVICE).to(dtype) 
        latte_model.eval() 

        print(f"Loading checkpoint from {model_ckpt_path}...")
        state_dict = torch.load(model_ckpt_path, map_location='cpu')
        if 'model' in state_dict: state_dict = state_dict['model']
        if 'ema' in state_dict: state_dict = state_dict['ema'] 
        
        model_state_dict = latte_model.state_dict()
        filtered_state_dict = {k: v for k, v in state_dict.items() if k in model_state_dict and v.shape == model_state_dict[k].shape}
        missing_keys, unexpected_keys = latte_model.load_state_dict(filtered_state_dict, strict=False)
        print(f"Checkpoint loaded. Missing keys: {missing_keys}, Unexpected keys: {unexpected_keys}")
        latte_model = latte_model.to(TPU_DEVICE)
        print("Diffusion Model loaded and moved to TPU.")

        scheduler = DDIMScheduler.from_config(args.scheduler_config if hasattr(args, 'scheduler_config') else 
                                            {"beta_schedule": "linear", "beta_start": 0.0001, "beta_end": 0.02, "num_train_timesteps": 1000})
        if hasattr(args, 'num_inference_steps'): scheduler.set_timesteps(args.num_inference_steps)
        else: scheduler.set_timesteps(50) 
        if hasattr(args, 'guidance_scale'): guidance_scale = args.guidance_scale 
        else: guidance_scale = 7.5

        print("Processing prompt...")
        text_output = text_encoder(prompt_text)
        prompt_embeds = text_output['prompt_embeds']
        text_padding_mask = text_output['text_padding_mask']

        num_frames = args.num_frames
        height, width = args.image_size, args.image_size
        latent_height, latent_width = height // vae.scale_factor, width // vae.scale_factor
        
        latents = torch.randn(1, vae.config.latent_channels, num_frames, latent_height, latent_width, device=TPU_DEVICE, dtype=dtype)

        print("Starting denoising loop...")
        with torch.no_grad(): 
            for t in scheduler.timesteps:
                print(f"  Step {t.item()}")
                latent_model_input = torch.cat([latents] * 2)
                t_single = torch.tensor([t], device=TPU_DEVICE, dtype=torch.long) 
                
                uncond_text_output = text_encoder("") 
                uncond_prompt_embeds = uncond_text_output['prompt_embeds']
                cfg_prompt_embeds = torch.cat([uncond_prompt_embeds, prompt_embeds])
                cfg_text_padding_mask = torch.cat([uncond_text_output['text_padding_mask'], text_padding_mask])

                noise_pred = latte_model(latent_model_input, t_single, cfg_prompt_embeds, cfg_text_padding_mask)

                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred_cfg = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
                
                latents = scheduler.step(noise_pred_cfg, t, latents).prev_sample
                xm.mark_step() 

        print("Denoising loop finished.")

        print("Decoding latents with VAE...")
        latents = 1.0 / vae.config.scaling_factor * latents
        video_frames_list = []
        for i in range(latents.size(2)): 
            current_latent_frame = latents[:, :, i, :, :]
            decoded_frame = vae.decode(current_latent_frame).sample
            video_frames_list.append(decoded_frame.cpu()) 
            xm.mark_step()
        
        video_tensor_cpu = torch.stack(video_frames_list, dim=2) 
        video_tensor_cpu = (video_tensor_cpu / 2 + 0.5).clamp(0, 1) 
        print("Latents decoded.")
        
        from torchvision.utils import save_image
        temp_image_path = os.path.join(VIDEO_OUTPUT_DIR_TPU, "temp_first_frame.png")
        save_image(video_tensor_cpu[0, :, 0, :, :], temp_image_path)
        print(f"Saved first frame to {temp_image_path}")

        prompt_slug_tpu = "".join(filter(str.isalnum, prompt_text.lower().replace(' ', '_')))[:30]
        timestamp = int(time.time())
        video_filename = f"{prompt_slug_tpu}_{timestamp}.mp4"
        video_path = os.path.join(VIDEO_OUTPUT_DIR_TPU, video_filename)

        try:
            import torchvision.io
            video_to_save = video_tensor_cpu[0].permute(1, 2, 3, 0).mul(255).byte().cpu() 
            torchvision.io.write_video(video_path, video_to_save, fps=args.fps if hasattr(args, 'fps') else 8)
            print(f"Video saved to {video_path} using torchvision.io")
        except Exception as e_tv:
            print(f"torchvision.io.write_video failed: {e_tv}. Falling back to ffmpeg if possible.")
            frame_dir = os.path.join(VIDEO_OUTPUT_DIR_TPU, f"frames_{timestamp}")
            os.makedirs(frame_dir, exist_ok=True)
            for i in range(video_tensor_cpu.size(2)):
                save_image(video_tensor_cpu[0, :, i, :, :], os.path.join(frame_dir, f"frame_{i:04d}.png"))
            ffmpeg_command = [
                'ffmpeg', '-y', '-framerate', str(args.fps if hasattr(args, 'fps') else 8),
                '-i', os.path.join(frame_dir, 'frame_%04d.png'), 
                '-c:v', 'libx264', '-pix_fmt', 'yuv420p', video_path
            ]
            try:
                subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                print(f"Video saved to {video_path} using ffmpeg.")
                import shutil
                shutil.rmtree(frame_dir) 
            except Exception as e_ff:
                print(f"ffmpeg also failed: {e_ff}")
                return temp_image_path, f"Video generation partially complete (first frame saved). Saving video file failed: {e_ff}"

        return video_path, f"Video generated: {video_filename}"

    except Exception as e:
        import traceback
        tb_str = traceback.format_exc()
        print(f"An error occurred during TPU video generation: {e}\n{tb_str}")
        return None, f"Error: {e}\n{tb_str}"

print("TPU inference function defined (Experimental).")

# Fallback stub function if the main one is too complex or fails early
def generate_video_tpu_stub(prompt_text, model_choice_tpu):
    print(f"Executing STUB for TPU generation: {prompt_text} with {model_choice_tpu}")
    time.sleep(2) # Simulate some work
    # Create a dummy output file for Gradio to display
    dummy_video_path = "/content/Open-Sora/samples_tpu/dummy_tpu_video.mp4"
    if not os.path.exists(VIDEO_OUTPUT_DIR_TPU):
        os.makedirs(VIDEO_OUTPUT_DIR_TPU)
    if not os.path.exists(dummy_video_path):
        try:
            subprocess.run([
                'ffmpeg', '-y', '-f', 'lavfi', '-i', 'testsrc=size=256x256:rate=10:duration=1',
                '-c:v', 'libx264', '-t', '1', '-pix_fmt', 'yuv420p', dummy_video_path
            ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e_ff_stub:
            print(f"Could not create dummy mp4 for stub: {e_ff_stub}")
            return None, f"TPU execution is a stub. Dummy video creation failed: {e_ff_stub}"
    return dummy_video_path, "TPU execution is a STUB. This is a DUMMY video. Full Open-Sora adaptation for TPU is a major task."
print("TPU inference function STUB also defined.")

### 5. Create Gradio Interface for TPU (Experimental)

In [ ]:
import gradio as gr

# Using the more complete generate_video_tpu if it's defined,
# otherwise fallback to the stub for safety in notebook execution flow.
inference_function_to_use = generate_video_tpu if 'generate_video_tpu' in globals() and callable(generate_video_tpu) else generate_video_tpu_stub

with gr.Blocks() as demo_tpu:
    gr.Markdown("## Open-Sora Text-to-Video Generator (TPU Experimental)")
    if inference_function_to_use == generate_video_tpu_stub:
        gr.Markdown("**Warning:** Running in STUB mode. The full `generate_video_tpu` function might have issues or is not fully implemented/debugged. This is primarily for testing the Gradio setup with TPU runtime.")
    else:
        gr.Markdown("**Note:** This is an experimental TPU version. Please report any issues. Ensure you have selected a TPU runtime in Colab.")
    with gr.Row():
        prompt_input_tpu = gr.Textbox(label="Enter your prompt", placeholder="A cat on a TPU...")
    with gr.Row():
        model_dropdown_tpu = gr.Dropdown(label="Select Model (TPU)", choices=list(MODEL_OPTIONS_TPU.keys()), value=list(MODEL_OPTIONS_TPU.keys())[0])
    with gr.Row():
        generate_button_tpu = gr.Button("Generate Video (TPU)")
    with gr.Row():
        video_output_tpu = gr.Video(label="Generated Video (TPU Output)")
    with gr.Row():
        status_output_tpu = gr.Textbox(label="Status (TPU)")

    generate_button_tpu.click(
        inference_function_to_use, 
        inputs=[prompt_input_tpu, model_dropdown_tpu],
        outputs=[video_output_tpu, status_output_tpu]
    )

print("Gradio interface for TPU defined.")

### 6. Setup Ngrok and Launch the Gradio App (TPU)

In [ ]:
# PASTE YOUR NGROK AUTHTOKEN HERE
ngrok_authtoken_tpu = "YOUR_NGROK_AUTHTOKEN" # Replace with your actual token

In [ ]:
from pyngrok import ngrok, conf

if ngrok_authtoken_tpu == "YOUR_NGROK_AUTHTOKEN" or not ngrok_authtoken_tpu:
    print("ERROR: Please set your Ngrok authtoken in the cell above!")
else:
    try:
        conf.get_default().auth_token = ngrok_authtoken_tpu
        # Kill any existing ngrok tunnels if any
        active_tunnels = ngrok.get_tunnels()
        for tunnel in active_tunnels:
            public_url = tunnel.public_url
            ngrok.disconnect(public_url)
            print(f"Killed existing tunnel: {public_url}")
        ngrok.kill() # Kill ngrok process if it's running to ensure a clean start
            
        public_url_tpu = ngrok.connect(7860) # Gradio default port
        print(f"Gradio App (TPU) is running at: {public_url_tpu}")
        print("Please open this URL in your browser.")
        # Use a thread for Gradio when running in Colab with ngrok to prevent blocking / issues
        import threading
        thread = threading.Thread(target=demo_tpu.launch, kwargs={'share': False, 'prevent_thread_lock': True})
        thread.start()
    except Exception as e:
        print(f"An error occurred with Ngrok or Gradio launch for TPU: {e}")
        print("Make sure your ngrok authtoken is correct and that you haven't exceeded ngrok's concurrent tunnel limits for your account type.")

### 7. Important Notes and Troubleshooting (TPU Version)
*   **TPU Runtime:** Ensure you have selected a TPU runtime in Colab. This notebook will not work with CPU or GPU runtimes.
*   **Experimental:** This is highly experimental. The `generate_video_tpu` function is a complex adaptation and may contain bugs or inefficiencies. The STUB function is a fallback if the main one fails early.
*   **XLA Compilation:** The first time you run an operation on TPU (like model inference), XLA will compile the graph. This can take some time. Subsequent runs of the same graph shape should be faster.
*   **Memory Limits:** TPUs also have memory limits. Large models or long videos might exceed these.
*   **Error Messages:** Pay close attention to error messages from PyTorch/XLA, Open-Sora, Gradio, or Ngrok. They often provide clues for debugging.
*   **Debugging `generate_video_tpu`:** This function is where most issues will likely arise. Debugging it involves:
    *   Checking that all model components are correctly moved to `TPU_DEVICE`.
    *   Ensuring tensor shapes are as expected by each component.
    *   Verifying that no CUDA-specific code paths are being hit.
    *   Using `xm.mark_step()` appropriately to manage XLA graph execution, especially in loops.
*   **Open-Sora Codebase:** A deep understanding of Open-Sora's internal code (especially `LatteT2V` model and inference scripts) would be necessary for a robust TPU port. This notebook provides a high-level attempt.
